In [ ]:
! pip install matplotlib numpy pandas scikit.learn scipy statsmodel joblib

# Import

In [ ]:
import argparse
import json
import logging
import pathlib
import subprocess
from copy import deepcopy
from datetime import datetime
from itertools import product
from pathlib import Path

import joblib
import matplotlib
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from joblib import Parallel, delayed
from scipy.stats import f_oneway
from sklearn.calibration import CalibratedClassifierCV
from sklearn.feature_selection import RFE
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score,
    f1_score,
    log_loss,
    precision_score,
    recall_score,
    roc_auc_score,
)
from sklearn.model_selection import train_test_split, RepeatedStratifiedKFold
from sklearn.preprocessing import StandardScaler
from sklearn.svm import SVC
from statsmodels.stats.multitest import multipletests

matplotlib.use("Agg")
logging.basicConfig(
    level=logging.DEBUG,
    format="%(asctime)s - %(name)s - %(levelname)s - %(message)s",
)
logger = logging.getLogger("project")

__version__ = "0.1.0"

CLINICAL_COLS = ["BM.PB", "Gender", "Source", "tissue.mf"]

SEED = 42

# Init

In [ ]:
def _git_suffix() -> str:
    """
    Add the necessary information to the version string.
    """
    # pylint: disable=broad-except
    kwargs = dict(cwd=pathlib.Path(__file__).parent, stderr=subprocess.DEVNULL)
    try:
        # Retrieve the git short sha to be appended to the base version string.
        args = ["git", "rev-parse", "--short", "HEAD"]
        sha = subprocess.check_output(args, **kwargs).decode().strip()
        suffix = f"+g{sha}"
        # If we have uncommitted changes, append a `.dirty` to the version suffix.
        args = ["git", "diff", "--quiet"]
        if subprocess.call(args, stdout=subprocess.DEVNULL, **kwargs) != 0:
            suffix = f"{suffix}.dirty"
        return suffix
    except Exception:
        return ""


__version__ = f"{__version__}{_git_suffix()}"


# Utils

In [ ]:
def load_config_json(filepath: str | Path) -> dict:
    """
    Load and return a JSON configuration file.

    Args:
        filepath (str | Path): path to the JSON configuration file.

    Returns:
        dict: parsed configuration.

    Raises:
        FileNotFoundError: if the file does not exist.
        json.JSONDecodeError: if the file is not valid JSON.
    """
    filepath = Path(filepath)
    try:
        with open(filepath, encoding="utf-8") as f:
            config = json.load(f)
        logger.info("Config loaded: %s", filepath)
        return config
    except FileNotFoundError:
        logger.error("Config file not found: %s", filepath)
        raise
    except json.JSONDecodeError as exc:
        logger.error("Invalid JSON in config file %s: %s", filepath, exc)
        raise

def calculate_metrics(y_true, y_pred, y_prob, model):
    """
    Calculate and return a dictionary of evaluation metrics.

    Args:
        y_true (np.ndarray): True labels.
        y_pred (np.ndarray): Predicted labels.
        y_prob (np.ndarray): Predicted probabilities for the positive class.
        model (SVC or LogisticRegression): The trained model used for predictions.
    
    Returns:
        dict: Dictionary containing accuracy, loss, precision, recall, F1 score, and ROC AUC.
    """
    metrics = {
        "accuracy":  accuracy_score(y_true, y_pred),
        "loss":      log_loss(y_true, y_prob, labels=model.classes_),
        "precision": precision_score(y_true, y_pred, pos_label=model.classes_[1], zero_division=0),
        "recall":    recall_score(y_true, y_pred, pos_label=model.classes_[1], zero_division=0),
        "f1_score":  f1_score(y_true, y_pred, pos_label=model.classes_[1], zero_division=0),
        "roc_auc":   roc_auc_score(y_true, y_prob[:, 1], labels=model.classes_),
    }
    return metrics

# Plotting

In [ ]:
def _draw_heatmap(ax, corr, labels, title, annotate=True):
    """
    Draw a heatmap of the correlation matrix with annotations.

    Args:
        ax: matplotlib axis to draw on.
        corr: 2D array of correlation coefficients.
        labels: list of variable names for axes.
        title: title of the plot.
    
    Returns:
        im: image object from imshow (for colorbar).
    """
    im = ax.imshow(corr, vmin=-1, vmax=1, cmap="RdBu_r", aspect="auto")
    ax.set_xticks(range(len(labels)))
    ax.set_yticks(range(len(labels)))
    ax.set_xticklabels(labels, rotation=45, ha="right", fontsize=8)
    ax.set_yticklabels(labels, fontsize=8)
    ax.set_title(title, fontsize=11)
    if annotate:
        for i in range(len(labels)):
            for j in range(len(labels)):
                val = corr[i, j]
                color = "white" if abs(val) > 0.6 else "black"
                ax.text(j, i, f"{val:.2f}", ha="center", va="center",
                        fontsize=6, color=color)
    return im


def plot_correlations(
    df: pd.DataFrame,
    output_dir: str | Path,
    max_vars: int = 7139,
    plot_name: str = "",
) -> None:
    """
    Plot Pearson correlation matrices for jet and track variables.

    Args:
        df (DataFrame): DataFrame of data file.
        output_dir (str | Path): Directory where PDFs are saved.
        max_vars (int): Maximum number of variables to plot (default: ``7139``).
        plot_name (str): Optional suffix for the plot filename (default: "").

    Warnings:
        If more than `max_vars` numeric columns are found, only the first `max_vars`
        are plotted (to avoid unreadable / oversized figures).
    """
    logger.info("Plotting correlations ...")
    output_dir = Path(output_dir)
    output_dir.mkdir(parents=True, exist_ok=True)

    numeric_df = df.select_dtypes(include="number")
    if numeric_df.empty:
        logger.warning("No numeric columns found. Skipping correlation plot.")
        return

    if numeric_df.shape[1] > max_vars:
        logger.warning(
            "%d numeric columns found. Plotting the first %d only.",
            numeric_df.shape[1], max_vars,
        )
        numeric_df = numeric_df.iloc[:, :max_vars]

    corr_mat = numeric_df.corr(method="pearson")
    labels   = numeric_df.columns.tolist()
    n        = len(labels)

    fig, ax = plt.subplots(figsize=(min(20, max(8, n * 0.55)), min(20, max(7, n * 0.55))))
    im = _draw_heatmap(ax, corr_mat.values, labels, "Variables - Correlation", annotate=(n <= 40))
    if n > 40:
        ax.set_xticks([])
        ax.set_yticks([])
    fig.colorbar(im, ax=ax, fraction=0.046, pad=0.04)
    fig.tight_layout()
    filename = "correlation_matrix" + (f"_{plot_name}" if plot_name else "") + ".pdf"
    out = output_dir / filename
    fig.savefig(out)
    plt.close(fig)

    logger.info("Saved: %s", out)


def plot_norm_stats(
    norm_stats: dict[str, np.ndarray],
    feature_names: list[str],
    output_dir: str | Path,
    max_features: int = 100,
) -> None:
    """
    Plot per-feature mean and standard deviation used for normalization.

    NaN entries in ``norm_stats`` (e.g. columns excluded from scaling) are
    skipped automatically.

    Args:
        norm_stats (dict): dict with keys ``"mean"``, ``"sigma"`` (np.ndarray,
            aligned with ``feature_names``).
        feature_names (list[str]): column names, same order/length as the
            arrays in ``norm_stats``.
        output_dir (str | Path): directory where the PDF is saved.
        max_features (int): if more than this many valid features are found,
            only the first ``max_features`` are plotted (avoids unreadable /
            oversized figures).
    """
    output_dir = Path(output_dir)
    output_dir.mkdir(parents=True, exist_ok=True)
    logger.info("Plotting normalization statistics ...")

    mean  = np.asarray(norm_stats["mean"])
    sigma = np.asarray(norm_stats["sigma"])

    mask = np.isfinite(mean) & np.isfinite(sigma)
    if not mask.any():
        logger.warning("No valid (non-NaN) normalization stats to plot.")
        return

    names = np.array(feature_names)[mask]
    mean  = np.sort(mean[mask])
    sigma = np.sort(sigma[mask])

    if len(names) > max_features:
        logger.warning(
            "%d normalized features found; plotting only the first %d.",
            len(names), max_features,
        )
        names, mean, sigma = names[:max_features], mean[:max_features], sigma[:max_features]

    n = len(names)
    x = np.arange(n)

    fig, axes = plt.subplots(2, 1, figsize=(min(24, max(8, n * 0.3)), 8), sharex=True)

    ax = axes[0]
    ax.bar(x, mean, color="steelblue")
    ax.set_ylabel("Mean", fontsize=12)
    ax.set_title("Per-feature normalization statistics (training set)", fontsize=13)
    if n > 40:
        ax.set_xticks([])
    ax.grid(True, alpha=0.3)

    ax = axes[1]
    ax.bar(x, sigma, color="darkorange")
    ax.set_ylabel("Std. dev.", fontsize=12)
    ax.set_xticks(x)
    ax.set_xticklabels(names, rotation=90, fontsize=6)
    if n > 40:
        ax.set_xticks([])
    ax.grid(True, alpha=0.3)

    fig.tight_layout()
    out = output_dir / "norm_stats.pdf"
    fig.savefig(out)
    plt.close(fig)

    logger.info("Saved: %s", out)


def plot_inclusion_probabilities(
    rfe_inclusion_prob: np.ndarray,
    fdr_inclusion_prob: np.ndarray,
    lasso_inclusion_prob: np.ndarray,
    output_dir: str | Path,
) -> None:
    """
    Plot the full distribution of feature inclusion probabilities
    (across all genes) for RFE, FDR and Lasso selection, plus a ranked
    comparison of the top genes.
 
    Args:
        rfe_inclusion_prob (np.ndarray): inclusion probability per feature, RFE.
        fdr_inclusion_prob (np.ndarray): inclusion probability per feature, FDR.
        lasso_inclusion_prob (np.ndarray): inclusion probability per feature, Lasso.
        output_dir (str | Path): directory where PDFs are saved.
    """
    output_dir = Path(output_dir)
    output_dir.mkdir(parents=True, exist_ok=True)
    logger.info("Plotting full inclusion-probability map ...")
 
    # 1. histograms: how many genes at each stability level
    fig, axes = plt.subplots(1, 3, figsize=(12, 5))
 
    bins = np.linspace(0, 1, 21)
    ax = axes[0]
    ax.hist(rfe_inclusion_prob, bins=bins, color="steelblue", edgecolor="k", alpha=0.8)
    ax.set_title("SVM-RFE: inclusion probability distribution")
    ax.set_xlabel("Inclusion probability")
    ax.set_ylabel("Number of genes")
    ax.set_yscale("log")
    ax.grid(True, alpha=0.3)
 
    ax = axes[1]
    ax.hist(fdr_inclusion_prob, bins=bins, color="darkorange", edgecolor="k", alpha=0.8)
    ax.set_title("FDR (BH): inclusion probability distribution")
    ax.set_xlabel("Inclusion probability")
    ax.set_yscale("log")
    ax.grid(True, alpha=0.3)

    ax = axes[2]
    ax.hist(lasso_inclusion_prob, bins=bins, color="green", edgecolor="k", alpha=0.8)
    ax.set_title("Lasso: inclusion probability distribution")
    ax.set_xlabel("Inclusion probability")
    ax.set_yscale("log")
    ax.grid(True, alpha=0.3)
 
    fig.tight_layout()
    out = output_dir / "inclusion_prob_histograms.pdf"
    fig.savefig(out)
    plt.close(fig)
    logger.info("Saved: %s", out)
 
    # 2. sorted rank plot: shows the "cliff" between stable/unstable genes
    sorted_rfe   = np.sort(rfe_inclusion_prob)[::-1]
    sorted_fdr   = np.sort(fdr_inclusion_prob)[::-1]
    sorted_lasso = np.sort(lasso_inclusion_prob)[::-1]
    fig, ax = plt.subplots(figsize=(10, 5))
    x = np.arange(len(sorted_rfe))
    ax.plot(x, sorted_rfe,   label="SVM-RFE",  color="steelblue",  linewidth=1.5)
    ax.plot(x, sorted_fdr,   label="FDR (BH)", color="darkorange", linewidth=1.5, alpha=0.8)
    ax.plot(x, sorted_lasso, label="Lasso",    color="green",      linewidth=1.5, alpha=0.8)
    ax.axhline(0.5, color="gray", linestyle="--", linewidth=1, label="50% threshold")
    ax.set_xlabel("Gene rank (sorted by inclusion prob.)")
    ax.set_ylabel("Inclusion probability")
    ax.set_title("Ranked inclusion probabilities across all genes")
    ax.legend()
    ax.grid(True, alpha=0.3)
 
    fig.tight_layout()
    out = output_dir / "inclusion_prob_ranked.pdf"
    fig.savefig(out)
    plt.close(fig)
    logger.info("Saved: %s", out)


def plot_bias_comparison(
    results: pd.DataFrame,
    output_dir: str | Path,
) -> None:
    """
    Plot the comparison between "biased" (external feature selection) and
    "unbiased" (internal feature selection) in terms of accuracy and loss
    as a function of the number of selected genes, replicating Figure 2 of
    Ambroise & McLachlan (2002).

    Args:
        results (pd.DataFrame): DataFrame containing the metrics of accuracy and loss.
        chance_level (float): expected accuracy for random case (1/n_classes).
        output_dir (str | Path): directory where to save the PDF.
    """
    metrics_dir = Path(output_dir) / "metrics"
    metrics_dir.mkdir(parents=True, exist_ok=True)
    logger.info("Plotting bias comparison ...")

    idx  = results.groupby("n_genes")["unbiased_rfe_val_loss"].idxmin()
    best = results.loc[idx].sort_values("n_genes")

    def _plot_metric(ax, metric_name, title):
        try:
            name = {
                "accuracy":  "acc",
                "loss":      "loss",
                "precision": "precision",
                "recall":    "recall",
                "F1 score":  "f1_score",
                "ROC AUC":   "roc_auc",
            }[metric_name]
        except KeyError:
            raise ValueError(f"Unknown metric name: {metric_name}")

        ax.plot(best["n_genes"], best[f"biased_rfe_train_{name}"],
                marker="o", ls="-.", color="crimson", label="Train (biased)")
        ax.plot(best["n_genes"], best[f"biased_rfe_val_{name}"],
                marker="o", ls="-", color="crimson", label="Validation (biased)")
        ax.plot(best["n_genes"], best[f"unbiased_rfe_train_{name}"],
                marker="o", ls="-.", color="steelblue", label="Train (unbiased)")
        ax.plot(best["n_genes"], best[f"unbiased_rfe_val_{name}"],
                marker="o", ls="-", color="steelblue", label="Validation (unbiased)")
        ax.set_xlabel("Number of selected genes")
        ax.set_ylabel(f"Average {metric_name} (CV)")
        ax.set_xscale("log", base=2)
        ax.set_title(title)
        ax.legend()
        ax.grid()

    fig, axes = plt.subplots(1, 2,figsize=(15, 5))

    ax = axes[0]
    _plot_metric(ax, "accuracy", "RFE")
    
    ax = axes[1]
    _plot_metric(ax, "accuracy", "Lasso")

    fig.suptitle("Selection bias: biased vs unbiased")
    fig.tight_layout()
    fig.savefig(metrics_dir / "accuracy_bias_comparison.pdf")
    plt.close(fig)


    fig, axes = plt.subplots(1, 2, figsize=(15, 5))

    ax = axes[0]
    _plot_metric(ax, "loss", "RFE")
    
    ax = axes[1]
    _plot_metric(ax, "loss", "Lasso")

    fig.suptitle("Selection bias: biased vs unbiased")
    fig.tight_layout()
    fig.savefig(metrics_dir / "loss_bias_comparison.pdf")
    plt.close(fig)


    fig, axes = plt.subplots(1, 2, figsize=(15, 5))

    ax = axes[0]
    _plot_metric(ax, "precision", "RFE")
    
    ax = axes[1]
    _plot_metric(ax, "precision", "Lasso")

    fig.suptitle("Selection bias: biased vs unbiased")
    fig.tight_layout()
    fig.savefig(metrics_dir / "precision_bias_comparison.pdf")
    plt.close(fig)

    fig, axes = plt.subplots(1, 2, figsize=(15, 5))

    ax = axes[0]
    _plot_metric(ax, "recall", "RFE")
    
    ax = axes[1]
    _plot_metric(ax, "recall", "Lasso")

    fig.suptitle("Selection bias: biased vs unbiased")
    fig.tight_layout()
    fig.savefig(metrics_dir / "recall_bias_comparison.pdf")
    plt.close(fig)

    fig, axes = plt.subplots(1, 2, figsize=(15, 5))

    ax = axes[0]
    _plot_metric(ax, "F1 score", "RFE")
    
    ax = axes[1]
    _plot_metric(ax, "F1 score", "Lasso")

    fig.suptitle("Selection bias: biased vs unbiased")
    fig.tight_layout()
    fig.savefig(metrics_dir / "f1_score_bias_comparison.pdf")
    plt.close(fig)

    fig, axes = plt.subplots(1, 2, figsize=(15, 5))

    ax = axes[0]
    _plot_metric(ax, "ROC AUC", "RFE")
    
    ax = axes[1]
    _plot_metric(ax, "ROC AUC", "Lasso")

    fig.suptitle("Selection bias: biased vs unbiased")
    fig.tight_layout()
    fig.savefig(metrics_dir / "roc_auc_bias_comparison.pdf")
    plt.close(fig)

    logger.info("Saved: %s", metrics_dir)


# Preprocess

In [ ]:
def compute_normalization_stats(
    X: pd.DataFrame,
) -> dict[str, np.ndarray]:
    """
    Compute per-feature mean std on the **training set only**.

    Statistics are computed only on numerical columns and exclusively on the training
    set to prevent data leakage.

    Args:
        X (pd.DataFrame): Train dataset dataframe.

    Returns:
        dict with keys ``"mean"``, ``"sigma"`` (each a ``np.ndarray``).
    """
    all_cols = X.columns.tolist()
    num_cols = X.select_dtypes(include=["number"]).columns.tolist()

    scaler = StandardScaler()
    scaler.fit(X[num_cols])

    mean  = np.full(len(all_cols), np.nan)
    sigma = np.full(len(all_cols), np.nan)

    numeric_positions = [all_cols.index(c) for c in num_cols]
    mean[numeric_positions]  = scaler.mean_
    sigma[numeric_positions] = scaler.scale_

    stats = {"mean": mean, "sigma": sigma}
    logger.info("Normalization stats computed on %s entries (%d numeric cols).",
                f"{len(X):,}", len(num_cols))
    return stats


def run_preprocess(df: pd.DataFrame, config: dict) -> None:
    """
    Run the full preprocessing pipeline.

    Reads all settings from ``config`` (already-parsed)
    and performs all preprocessing steps, including train/val/test
    splitting and cropping of clinical columns.

    Config keys read:
        - `data`:
            - `cv_fraction` (float)
            - `test_fraction` (float)
            - `shuffle` (bool, default ``False``)
            - `split_seed` (int, default ``42``)

    Args:
        df (pd.DataFrame): Full dataset dataframe.
        config (dict): Full configuration dict.

    Warnings:
        ValueError: If the train + val / test fractions do not sum to 1.
    """
    logger.info("=== Preprocess ===")

    df = df.drop(columns=["Samples"], errors="ignore")
    X = df.drop(columns=["cancer"] + CLINICAL_COLS)
    y = df["cancer"]

    y = y.replace({"allB": "ALL", "allT": "ALL"})

    # 1. load configuration
    data_config = config["data"]

    cv_frac   = data_config["cv_fraction"]
    test_frac = data_config["test_fraction"]
    shuffle   = data_config.get("shuffle", True)
    seed      = data_config.get("split_seed", 42)

    total = cv_frac + test_frac
    if abs(total - 1.) > 1e-6:
        logger.warning(
            "Fractions sum to %.6f, normalizing to 1.0", total
        )
        cv_frac   /= total
        test_frac /= total

    # 2. train + val / test split
    X_cv, X_test, y_cv, y_test = train_test_split(
        X, y,
        train_size   = cv_frac,
        random_state = seed,
        shuffle      = shuffle,
        stratify     = y,
    )

    logger.info("Preprocessing complete.")

    return X_cv, X_test, y_cv, y_test


# Feature selection

In [ ]:
def statistical_fdr_selection(
    X: pd.DataFrame,
    y: pd.Series,
    alpha: float = 0.05,
):
    """
    Performs a one-way ANOVA for each gene across all classes and applies
    Benjamini-Hochberg FDR correction.
    Returns a boolean mask of significant genes and the q-values.

    Args:
        X (pd.DataFrame): Feature matrix.
        y (pd.Series): Target vector.
        alpha (float): Significance level for FDR correction (default: ``0.05``).
    
    Returns:
        reject (np.ndarray): Boolean mask of significant genes.
        q_values (np.ndarray): Array of q-values for each gene.
    """
    p_values = []
    groups = [X[y == cls] for cls in y.unique()]
    for col in X.columns:
        samples = [g[col].values for g in groups]
        _, p = f_oneway(*samples)
        p_values.append(p if not np.isnan(p) else 1.0)

    p_values = np.array(p_values)
    reject, q_values, _, _ = multipletests(p_values, alpha=alpha, method="fdr_bh")
    return reject, q_values


def rfe_svm_selection(
    X_scaled: np.ndarray,
    y: pd.Series,
    n_genes: int = 30,
    kernel: str = "linear"
):
    """
    RFE with Linear SVM (inspired by Guyon et al., 2002).
    
    Args:
        X_scaled (np.ndarray): Scaled feature matrix.
        y (pd.Series): Target vector.
        n_genes (int): Number of features to select (default: ``30``).
    
    Returns:
        selector.support_ (np.ndarray): Boolean mask of selected features.
    """
    estimator = SVC(kernel=kernel, random_state=SEED)
    selector = RFE(estimator=estimator, n_features_to_select=n_genes, step=0.1)
    selector.fit(X_scaled, y)
    return selector.support_


def lasso_selection(
    X: np.ndarray,
    y: pd.Series,
    C: float = 0.1,
) -> np.ndarray:
    """
    L1-penalized logistic regression (Lasso) feature selection.

    A third, independent selection criterion alongside FDR (statistical)
    and SVM-RFE (wrapper ML method): genes with a non-zero coefficient in
    at least one class are considered "selected". With 3 classes, `coef_`
    has shape (n_classes, n_features) under the multinomial scheme, so a
    feature is selected if ANY class has a non-zero weight for it.
    Smaller C -> stronger penalty -> sparser selection.

    Args:
        X (np.ndarray): feature matrix.
        y (pd.Series): target vector (binary or multiclass).
        C (float): inverse regularization strength (default 0.1).

    Returns:
        np.ndarray: boolean mask of selected features.
    """
    model = LogisticRegression(
        solver="saga",
        l1_ratio=1.0,          # 1.0 = L1, 0.0 = L2, (0.0, 1.0) = ElasticNet
        C=C,
        random_state=SEED,
        max_iter=10000,
    )
    model.fit(X, y)
    mask = np.any(model.coef_ != 0, axis=0)

    if not mask.any():
        logger.warning(
            "Lasso (C=%.4f) selected 0 features — falling back to top-1 by |coef|.", C
        )
        best = np.argmax(np.abs(model.coef_).sum(axis=0))
        mask = np.zeros(X.shape[1], dtype=bool)
        mask[best] = True

    return mask


# Cross validation

In [ ]:
def _run_one_fold_biased(X, y, train_idx, val_idx, kernel):
    X_train, y_train = X.iloc[train_idx], y.iloc[train_idx]
    X_val,   y_val   = X.iloc[val_idx],   y.iloc[val_idx]

    scaler = StandardScaler()
    X_train_scaled = pd.DataFrame(scaler.fit_transform(X_train),
                                  columns=X_train.columns,
                                  index=X_train.index)
    X_val_scaled   = pd.DataFrame(scaler.transform(X_val),
                                  columns=X_val.columns,
                                  index=X_val.index)

    model = CalibratedClassifierCV(SVC(kernel=kernel, random_state=SEED), ensemble=False)
    model.fit(X_train_scaled, y_train)

    train_preds, train_probs = model.predict(X_train_scaled), model.predict_proba(X_train_scaled)
    val_preds,   val_probs   = model.predict(X_val_scaled),   model.predict_proba(X_val_scaled)

    return {
        "train": calculate_metrics(y_train, train_preds, train_probs, model),
        "val": calculate_metrics(y_val, val_preds, val_probs, model),
    }

def _run_one_fold_unbiased(X, y, train_idx, val_idx, kernel, alpha, n_genes, C):
    X_train, y_train = X.iloc[train_idx], y.iloc[train_idx]
    X_val,   y_val   = X.iloc[val_idx],   y.iloc[val_idx]

    scaler = StandardScaler()
    X_train_scaled = pd.DataFrame(scaler.fit_transform(X_train),
                                  columns=X_train.columns,
                                  index=X_train.index)
    X_val_scaled   = pd.DataFrame(scaler.transform(X_val),
                                  columns=X_val.columns,
                                  index=X_val.index)

    fdr_mask, _ = statistical_fdr_selection(X_train_scaled, y_train, alpha=alpha)
    rfe_mask    = rfe_svm_selection(X_train_scaled, y_train, n_genes=n_genes)
    lasso_mask  = lasso_selection(X_train_scaled, y_train, C=C)

    X_train_sel_rfe   = X_train_scaled.iloc[:, rfe_mask]
    X_val_sel_rfe     = X_val_scaled.iloc[:, rfe_mask]
    X_train_sel_lasso = X_train_scaled.iloc[:, lasso_mask]
    X_val_sel_lasso   = X_val_scaled.iloc[:, lasso_mask]

    model_rfe   = CalibratedClassifierCV(SVC(kernel=kernel, random_state=SEED), ensemble=False)
    model_rfe.fit(X_train_sel_rfe, y_train)
    model_lasso = CalibratedClassifierCV(SVC(kernel=kernel, random_state=SEED), ensemble=False)
    model_lasso.fit(X_train_sel_lasso, y_train)

    logger.debug(
        "Fold feature counts - RFE: %d | FDR: %d | Lasso: %d",
        rfe_mask.sum(), fdr_mask.sum(), lasso_mask.sum()
    )

    return {
        "fdr_mask": fdr_mask, "rfe_mask": rfe_mask, "lasso_mask": lasso_mask,
        "rfe":   {
            "train": calculate_metrics(y_train, model_rfe.predict(X_train_sel_rfe),
                                       model_rfe.predict_proba(X_train_sel_rfe), model_rfe),
            "val":   calculate_metrics(y_val,   model_rfe.predict(X_val_sel_rfe),
                                       model_rfe.predict_proba(X_val_sel_rfe),   model_rfe),
        },
        "lasso": {
            "train": calculate_metrics(y_train, model_lasso.predict(X_train_sel_lasso),
                                       model_lasso.predict_proba(X_train_sel_lasso), model_lasso),
            "val":   calculate_metrics(y_val,   model_lasso.predict(X_val_sel_lasso),
                                       model_lasso.predict_proba(X_val_sel_lasso),   model_lasso),
        },
    }


def cross_validation_biased(
    X: pd.DataFrame,
    y: pd.DataFrame,
    n_splits: int,
    n_repeats: int,
    parameters: list,
    kernel: str = "linear",
    n_jobs: int = -1,
):
    """
    Run cross-validation with external feature selection (RFE + Lasso) and training of a linear SVM.

    Args:
        X (pd.DataFrame): Feature matrix.
        y (pd.DataFrame): Target vector.
        n_splits (int): Number of splits for CV.
        n_repeats (int): Number of repeats for CV.
        parameters (list): List of parameters for feature selection and modeling
            (``num_features``, ``alpha``, ``C``).
        kernel (str, optional): Kernel type for SVM. (Default: ``linear``)
    
    Returns:
        train_metrics (list): List containing mean training accuracy and loss for RFE and Lasso.
        val_metrics (list): List containing mean validation accuracy and loss for RFE and Lasso.
    """
    n_genes, _, lasso_C = parameters

    # 1. feature selection
    # SVM-RFE
    rfe_mask         = rfe_svm_selection(X, y, n_genes=n_genes)
    X_selected_rfe   = X.iloc[:, rfe_mask]
    # Lasso
    lasso_mask       = lasso_selection(X, y, C=lasso_C)
    X_selected_lasso = X.iloc[:, lasso_mask]

    # 2. CV
    rskf = RepeatedStratifiedKFold(n_splits=n_splits, n_repeats=n_repeats, random_state=SEED)

    base = {"rfe": [], "lasso": []}
    train_metrics = {
        "accuracy":  deepcopy(base), "loss":    deepcopy(base),
        "precision": deepcopy(base), "recall":  deepcopy(base),
        "f1_score":  deepcopy(base), "roc_auc": deepcopy(base)
    }
    val_metrics   = {
        "accuracy":  deepcopy(base), "loss":    deepcopy(base),
        "precision": deepcopy(base), "recall":  deepcopy(base),
        "f1_score":  deepcopy(base), "roc_auc": deepcopy(base)
    }

    fold_results_rfe = Parallel(n_jobs=n_jobs)(
        delayed(_run_one_fold_biased)(X_selected_rfe, y, train_idx, val_idx, kernel)
        for train_idx, val_idx in rskf.split(X_selected_rfe, y)
    )

    for res in fold_results_rfe:
        for metric in train_metrics:
            train_metrics[metric]["rfe"].append(res["train"][metric])
            val_metrics[metric]["rfe"].append(res["val"][metric])
        
    fold_results_lasso = Parallel(n_jobs=n_jobs)(
        delayed(_run_one_fold_biased)(X_selected_lasso, y, train_idx, val_idx, kernel)
        for train_idx, val_idx in rskf.split(X_selected_lasso, y)
    )

    for res in fold_results_lasso:
        for metric in train_metrics:
            train_metrics[metric]["lasso"].append(res["train"][metric])
            val_metrics[metric]["lasso"].append(res["val"][metric])

    val_rfe_mean_accuracy    = float(np.mean(val_metrics["accuracy"]["rfe"]))
    val_rfe_mean_loss        = float(np.mean(val_metrics["loss"]["rfe"]))
    val_lasso_mean_accuracy  = float(np.mean(val_metrics["accuracy"]["lasso"]))
    val_lasso_mean_loss      = float(np.mean(val_metrics["loss"]["lasso"]))
    logger.info(
        "Biased CV completed. Mean Validation Metrics: RFE Accuracy = %.4f, RFE Loss = %.4f, "
        "Lasso Accuracy = %.4f, Lasso Loss = %.4f",
        val_rfe_mean_accuracy, val_rfe_mean_loss, val_lasso_mean_accuracy, val_lasso_mean_loss
    )

    return (
        {
            "rfe":   [float(np.mean(train_metrics[m]["rfe"]))   for m in train_metrics],
            "lasso": [float(np.mean(train_metrics[m]["lasso"])) for m in train_metrics],
        },
        {
            "rfe":   [float(np.mean(val_metrics[m]["rfe"]))   for m in val_metrics],
            "lasso": [float(np.mean(val_metrics[m]["lasso"])) for m in val_metrics],
        },
    )


def cross_validation_unbiased(
    X: pd.DataFrame,
    y: pd.DataFrame,
    n_splits: int,
    n_repeats: int,
    parameters: list,
    kernel: str = "linear",
    n_jobs: int = -1,
):
    """
    Run cross-validation with internal feature selection (RFE + RFE + Lasso)
    and training of a linear SVM.

    Args:
        X (pd.DataFrame): Feature matrix.
        y (pd.DataFrame): Target vector.
        n_splits (int): Number of splits for CV.
        n_repeats (int): Number of repeats for CV.
        parameters (list): List of parameters for feature selection and modeling
            (``num_features``, ``alpha``, ``C``).
        kernel (str, optional): Kernel type for SVM. (Default: ``linear``)

    Returns:
        inclusion_probs (np.ndarray): Array of inclusion probabilities for each gene.
        train_metrics (list): List containing mean training accuracy and loss for RFE and Lasso.
        val_metrics (list): List containing mean validation accuracy and loss for RFE and Lasso
    """
    rskf = RepeatedStratifiedKFold(n_splits=n_splits, n_repeats=n_repeats, random_state=SEED)

    n_genes, fdr_alpha, lasso_C = parameters

    rfe_selection_counts   = np.zeros(X.shape[1])
    fdr_selection_counts   = np.zeros(X.shape[1])
    lasso_selection_counts = np.zeros(X.shape[1])

    base = {"rfe": [], "lasso": []}
    train_metrics = {
        "accuracy":  deepcopy(base), "loss":    deepcopy(base),
        "precision": deepcopy(base), "recall":  deepcopy(base),
        "f1_score":  deepcopy(base), "roc_auc": deepcopy(base)
    }
    val_metrics   = {
        "accuracy":  deepcopy(base), "loss":    deepcopy(base),
        "precision": deepcopy(base), "recall":  deepcopy(base),
        "f1_score":  deepcopy(base), "roc_auc": deepcopy(base)
    }

    fold_results = Parallel(n_jobs=n_jobs)(
        delayed(_run_one_fold_unbiased)(X, y, train_idx, val_idx, kernel,
                                        fdr_alpha, n_genes, lasso_C)
        for train_idx, val_idx in rskf.split(X, y)
    )

    for res in fold_results:
        fdr_selection_counts   += res["fdr_mask"]
        rfe_selection_counts   += res["rfe_mask"]
        lasso_selection_counts += res["lasso_mask"]
        for metric in train_metrics:
            train_metrics[metric]["rfe"].append(res["rfe"]["train"][metric])
            val_metrics[metric]["rfe"].append(res["rfe"]["val"][metric])
            train_metrics[metric]["lasso"].append(res["lasso"]["train"][metric])
            val_metrics[metric]["lasso"].append(res["lasso"]["val"][metric])

    # calculate inclusion probabilities
    total_runs = n_splits * n_repeats
    rfe_inclusion_prob   = rfe_selection_counts   / total_runs
    fdr_inclusion_prob   = fdr_selection_counts   / total_runs
    lasso_inclusion_prob = lasso_selection_counts / total_runs

    # top 10 biomarkers for stability (SVM-RFE)
    top_rfe_idx = np.argsort(rfe_inclusion_prob)[::-1][:10]
    logger.info("=== TOP 10 BIOMARKS FOR STABILITY (SVM-RFE) ===")
    for idx in top_rfe_idx:
        logger.info("Gene: %-16s | Probability of Inclusion: %.2f",
                    X.columns[idx], rfe_inclusion_prob[idx])
    # top 10 biomarkers for stability (SVM-Lasso)
    top_lasso_idx = np.argsort(lasso_inclusion_prob)[::-1][:10]
    logger.info("=== TOP 10 BIOMARKS FOR STABILITY (SVM-Lasso) ===")
    for idx in top_lasso_idx:
        logger.info("Gene: %-16s | Probability of Inclusion: %.2f",
                    X.columns[idx], lasso_inclusion_prob[idx])

    # comparison of inclusion probabilities between methods
    all_three = np.sum(
        (rfe_inclusion_prob   > 0.5) &
        (fdr_inclusion_prob   > 0.5) &
        (lasso_inclusion_prob > 0.5)
    )
    logger.info(f"Stable genes (>50%) in common between FDR, RFE and Lasso: {all_three}")

    val_rfe_mean_accuracy    = float(np.mean(val_metrics["accuracy"]["rfe"]))
    val_rfe_mean_loss        = float(np.mean(val_metrics["loss"]["rfe"]))
    val_lasso_mean_accuracy  = float(np.mean(val_metrics["accuracy"]["lasso"]))
    val_lasso_mean_loss      = float(np.mean(val_metrics["loss"]["lasso"]))
    logger.info(
        "Unbiased CV completed. Mean Validation Metrics: RFE Accuracy = %.4f, RFE Loss = %.4f, "
        "Lasso Accuracy = %.4f, Lasso Loss = %.4f\n",
        val_rfe_mean_accuracy, val_rfe_mean_loss, val_lasso_mean_accuracy, val_lasso_mean_loss
    )

    return (
        [rfe_inclusion_prob, fdr_inclusion_prob, lasso_inclusion_prob],
        {
            "rfe":   [float(np.mean(train_metrics[m]["rfe"]))   for m in train_metrics],
            "lasso": [float(np.mean(train_metrics[m]["lasso"])) for m in train_metrics],
        },
        {
            "rfe":   [float(np.mean(val_metrics[m]["rfe"]))   for m in val_metrics],
            "lasso": [float(np.mean(val_metrics[m]["lasso"])) for m in val_metrics],
        },
    )


# Bias experiment

In [ ]:
METRIC_ORDER = ["accuracy", "loss", "precision", "recall", "f1_score", "roc_auc"]

def _unpack_metrics(flat_list):
    """Converte una lista piatta [rfe_acc, rfe_loss, ..., lasso_acc, ...] in dict."""
    rfe   = dict(zip(METRIC_ORDER, flat_list[:6], strict=True))
    lasso = dict(zip(METRIC_ORDER, flat_list[6:], strict=True))
    return {"rfe": rfe, "lasso": lasso}


def _run_one_combination(X, y, n_splits, n_repeats, gene, alpha, C, inner_n_jobs):
    logger.info("=== n_genes = %d | alpha = %f | C = %f ===", gene, alpha, C)
    train_biased_metric, val_biased_metric = cross_validation_biased(
        X, y,
        n_splits=n_splits,
        n_repeats=n_repeats,
        parameters=[gene, alpha, C],
        n_jobs=inner_n_jobs,
    )
    unbiased_inclusion_prob, train_unbiased_metric, val_unbiased_metric = cross_validation_unbiased(
        X, y,
        n_splits=n_splits,
        n_repeats=n_repeats,
        parameters=[gene, alpha, C],
        n_jobs=inner_n_jobs,
    )
    return {
        "parameter":               [gene, alpha, C],
        "train_biased":            train_biased_metric,
        "val_biased":              val_biased_metric,
        "unbiased_inclusion_prob": unbiased_inclusion_prob,
        "train_unbiased":          train_unbiased_metric,
        "val_unbiased":            val_unbiased_metric,
    }

def run_bias_experiment(
    X: pd.DataFrame,
    y: pd.Series,
    n_splits: int,
    n_repeats: int,
    parameters: list,
    outer_n_jobs: int = -1,
    inner_n_jobs: int = 1,
):
    """
    Runs the selection bias experiment comparing biased and unbiased feature
    selection methods (RFE and Lasso) on the provided dataset.

    Args:
        X (pd.DataFrame): Feature matrix.
        y (pd.Series): Target vector.
        n_splits (int): Number of splits for CV.
        n_repeats (int): Number of repeats for CV.
        parameters (list, optional): List of parameters for feature selection and modeling
            (``genes``, ``alpha``, ``C``).

    Returns:
        results (pd.DataFrame): DataFrame containing the results of the biase
            and unbiased feature selection methods.
        unbiased_inclusion_probs (list): List of inclusion probabilities for
            each gene from the unbiased method
    """
    logger.info("=== Selection bias experiment ===")
    genes, alphas, Cs = parameters[0], parameters[1], parameters[2]

    n_classes = y.nunique()
    chance_level = 1. / n_classes
    logger.info("Chance level (random guessing): %.4f", chance_level)

    combo_results = Parallel(n_jobs=outer_n_jobs)(
        delayed(_run_one_combination)(X, y, n_splits, n_repeats, gene,
                                      alpha, C, inner_n_jobs)
        for gene, alpha, C in product(genes, alphas, Cs)
    )

    base = {
        "rfe":   {"train": [], "val": []},
        "lasso": {"train": [], "val": []}
    }
    biased_metrics = {
        "accuracy":  deepcopy(base), "loss":    deepcopy(base),
        "precision": deepcopy(base), "recall":  deepcopy(base),
        "f1_score":  deepcopy(base), "roc_auc": deepcopy(base)
    }
    unbiased_metrics = {
        "accuracy":  deepcopy(base), "loss":    deepcopy(base),
        "precision": deepcopy(base), "recall":  deepcopy(base),
        "f1_score":  deepcopy(base), "roc_auc": deepcopy(base)
    }
    unbiased_inclusion_probs = []
    parameter                = []

    for res in combo_results:
        for split in ("train", "val"):
            unpacked = _unpack_metrics(res[f"{split}_biased"])
            for metric in METRIC_ORDER:
                biased_metrics[metric]["rfe"][split].append(unpacked["rfe"][metric])
                biased_metrics[metric]["lasso"][split].append(unpacked["lasso"][metric])
            unpacked = _unpack_metrics(res[f"{split}_unbiased"])
            for metric in METRIC_ORDER:
                unbiased_metrics[metric]["rfe"][split].append(unpacked["rfe"][metric])
                unbiased_metrics[metric]["lasso"][split].append(unpacked["lasso"][metric])

        unbiased_inclusion_probs.append(res["unbiased_inclusion_prob"])
        parameter.append(res["parameter"])

    results = pd.DataFrame({
        "n_genes": [p[0] for p in parameter],
        "alpha":   [p[1] for p in parameter],
        "C":       [p[2] for p in parameter],

        "biased_rfe_train_acc":           biased_metrics["accuracy"]["rfe"]["train"],
        "biased_rfe_val_acc":             biased_metrics["accuracy"]["rfe"]["val"],

        "biased_rfe_train_loss":          biased_metrics["loss"]["rfe"]["train"],
        "biased_rfe_val_loss":            biased_metrics["loss"]["rfe"]["val"],

        "biased_rfe_train_precision":     biased_metrics["precision"]["rfe"]["train"],
        "biased_rfe_val_precision":       biased_metrics["precision"]["rfe"]["val"],

        "biased_rfe_train_recall":        biased_metrics["recall"]["rfe"]["train"],
        "biased_rfe_val_recall":          biased_metrics["recall"]["rfe"]["val"],

        "biased_rfe_train_f1_score":      biased_metrics["f1_score"]["rfe"]["train"],
        "biased_rfe_val_f1_score":        biased_metrics["f1_score"]["rfe"]["val"],

        "biased_rfe_train_roc_auc":       biased_metrics["roc_auc"]["rfe"]["train"],
        "biased_rfe_val_roc_auc":         biased_metrics["roc_auc"]["rfe"]["val"],

        "biased_lasso_train_acc":         biased_metrics["accuracy"]["lasso"]["train"],
        "biased_lasso_val_acc":           biased_metrics["accuracy"]["lasso"]["val"],

        "biased_lasso_train_loss":        biased_metrics["loss"]["lasso"]["train"],
        "biased_lasso_val_loss":          biased_metrics["loss"]["lasso"]["val"],

        "biased_lasso_train_precision":   biased_metrics["precision"]["lasso"]["train"],
        "biased_lasso_val_precision":     biased_metrics["precision"]["lasso"]["val"],

        "biased_lasso_train_recall":      biased_metrics["recall"]["lasso"]["train"],
        "biased_lasso_val_recall":        biased_metrics["recall"]["lasso"]["val"],

        "biased_lasso_train_f1_score":    biased_metrics["f1_score"]["lasso"]["train"],
        "biased_lasso_val_f1_score":      biased_metrics["f1_score"]["lasso"]["val"],

        "biased_lasso_train_roc_auc":     biased_metrics["roc_auc"]["lasso"]["train"],
        "biased_lasso_val_roc_auc":       biased_metrics["roc_auc"]["lasso"]["val"],

        "unbiased_rfe_train_acc":         unbiased_metrics["accuracy"]["rfe"]["train"],
        "unbiased_rfe_val_acc":           unbiased_metrics["accuracy"]["rfe"]["val"],

        "unbiased_rfe_train_loss":        unbiased_metrics["loss"]["rfe"]["train"],
        "unbiased_rfe_val_loss":          unbiased_metrics["loss"]["rfe"]["val"],

        "unbiased_rfe_train_precision":   unbiased_metrics["precision"]["rfe"]["train"],
        "unbiased_rfe_val_precision":     unbiased_metrics["precision"]["rfe"]["val"],

        "unbiased_rfe_train_recall":      unbiased_metrics["recall"]["rfe"]["train"],
        "unbiased_rfe_val_recall":        unbiased_metrics["recall"]["rfe"]["val"],

        "unbiased_rfe_train_f1_score":    unbiased_metrics["f1_score"]["rfe"]["train"],
        "unbiased_rfe_val_f1_score":      unbiased_metrics["f1_score"]["rfe"]["val"],

        "unbiased_rfe_train_roc_auc":     unbiased_metrics["roc_auc"]["rfe"]["train"],
        "unbiased_rfe_val_roc_auc":       unbiased_metrics["roc_auc"]["rfe"]["val"],

        "unbiased_lasso_train_acc":       unbiased_metrics["accuracy"]["lasso"]["train"],
        "unbiased_lasso_val_acc":         unbiased_metrics["accuracy"]["lasso"]["val"],

        "unbiased_lasso_train_loss":      unbiased_metrics["loss"]["lasso"]["train"],
        "unbiased_lasso_val_loss":        unbiased_metrics["loss"]["lasso"]["val"],

        "unbiased_lasso_train_precision": unbiased_metrics["precision"]["lasso"]["train"],
        "unbiased_lasso_val_precision":   unbiased_metrics["precision"]["lasso"]["val"],

        "unbiased_lasso_train_recall":    unbiased_metrics["recall"]["lasso"]["train"],
        "unbiased_lasso_val_recall":      unbiased_metrics["recall"]["lasso"]["val"],

        "unbiased_lasso_train_f1_score":  unbiased_metrics["f1_score"]["lasso"]["train"],
        "unbiased_lasso_val_f1_score":    unbiased_metrics["f1_score"]["lasso"]["val"],

        "unbiased_lasso_train_roc_auc":   unbiased_metrics["roc_auc"]["lasso"]["train"],
        "unbiased_lasso_val_roc_auc":     unbiased_metrics["roc_auc"]["lasso"]["val"],
    })

    logger.info("Bias experiment complete.")

    return results, unbiased_inclusion_probs


# Evaluate

In [ ]:
def evaluate(
    X_cv: pd.DataFrame,
    y_cv: pd.DataFrame,
    X_test: pd.DataFrame,
    y_test: pd.DataFrame,
    n_genes: int,
    rfe_inclusion_prob: np.ndarray,
    lasso_inclusion_prob: np.ndarray,
    kernel: str = "linear",
    plot_dir: str | Path = None,
    run_dir: str | Path = None
):
    """
    Evaluates the performance of a machine learning model on a test set using
    stable genes selected based on their inclusion probabilities from RFE.

    Args:
        X_cv (pd.DataFrame): Cross-validation feature matrix.
        y_cv (pd.DataFrame): Cross-validation target vector.
        X_test (pd.DataFrame): Test feature matrix.
        y_test (pd.DataFrame): Test target vector.
        n_genes (int): Number of features to select based on
            RFE inclusion probabilities.
        rfe_inclusion_prob (np.ndarray): Array of inclusion probabilities
            for each gene from RFE.
        lasso_inclusion_prob (np.ndarray): Array of inclusion probabilities
            for each gene from Lasso.
        plot_dir (str | Path): Directory to save correlation plots (optional, default is ``None``).
    """
    logger.info("=== Evaluate ===")

    stable_genes_mask_rfe = np.argsort(rfe_inclusion_prob)[::-1][:n_genes]
    stable_genes_mask_lasso = np.argsort(lasso_inclusion_prob)[::-1]
    stable_genes_mask_lasso = stable_genes_mask_lasso[
                              lasso_inclusion_prob[stable_genes_mask_lasso] != 0
                              ][:n_genes]

    stable_gene_names_rfe     = X_cv.columns[stable_genes_mask_rfe].tolist()
    df_stable_rfe             = X_cv[stable_gene_names_rfe].copy()
    df_stable_rfe["cancer"]   = y_cv.values
    stable_gene_names_lasso   = X_cv.columns[stable_genes_mask_lasso].tolist()
    df_stable_lasso           = X_cv[stable_gene_names_lasso].copy()
    df_stable_lasso["cancer"] = y_cv.values

    if plot_dir is not None:
        plot_correlations(
            df=df_stable_rfe.select_dtypes(include="number"),
            output_dir=plot_dir,
            plot_name="stable_genes_rfe"
        )
        plot_correlations(
            df=df_stable_lasso.select_dtypes(include="number"),
            output_dir=plot_dir,
            plot_name="stable_genes_lasso"
        )
        
    scaler = StandardScaler()
    X_cv   = scaler.fit_transform(X_cv)
    X_test = scaler.transform(X_test)

    final_model_rfe = CalibratedClassifierCV(SVC(kernel=kernel, random_state=SEED),
                                             ensemble=False)
    final_model_rfe.fit(X_cv[:, stable_genes_mask_rfe], y_cv)
    final_model_lasso = CalibratedClassifierCV(SVC(kernel=kernel, random_state=SEED),
                                               ensemble=False)
    final_model_lasso.fit(X_cv[:, stable_genes_mask_lasso], y_cv)

    train_preds_rfe   = final_model_rfe.predict(X_cv[:, stable_genes_mask_rfe])
    train_probs_rfe   = final_model_rfe.predict_proba(X_cv[:, stable_genes_mask_rfe])
    test_preds_rfe    = final_model_rfe.predict(X_test[:, stable_genes_mask_rfe])
    test_probs_rfe    = final_model_rfe.predict_proba(X_test[:, stable_genes_mask_rfe])
    train_preds_lasso = final_model_lasso.predict(X_cv[:, stable_genes_mask_lasso])
    train_probs_lasso = final_model_lasso.predict_proba(X_cv[:, stable_genes_mask_lasso])
    test_preds_lasso  = final_model_lasso.predict(X_test[:, stable_genes_mask_lasso])
    test_probs_lasso  = final_model_lasso.predict_proba(X_test[:, stable_genes_mask_lasso])

    train_rfe_metrics   = calculate_metrics(y_cv,   train_preds_rfe,
                                            train_probs_rfe,   final_model_rfe)
    test_rfe_metrics    = calculate_metrics(y_test, test_preds_rfe,
                                            test_probs_rfe,    final_model_rfe)
    train_lasso_metrics = calculate_metrics(y_cv,   train_preds_lasso,
                                            train_probs_lasso, final_model_lasso)
    test_lasso_metrics  = calculate_metrics(y_test, test_preds_lasso,
                                            test_probs_lasso,  final_model_lasso)

    joblib.dump(final_model_rfe, run_dir / "rfe_model.joblib")
    joblib.dump(final_model_lasso, run_dir / "lasso_model.joblib")

    logger.info(
        "Train RFE   -> Acc: %.4f | Loss: %.4f | Prec: %.4f | Rec: %.4f | F1: %.4f | AUC: %.4f",
        train_rfe_metrics["accuracy"], train_rfe_metrics["loss"], train_rfe_metrics["precision"],
        train_rfe_metrics["recall"], train_rfe_metrics["f1_score"], train_rfe_metrics["roc_auc"]
    )
    logger.info(
        "Test RFE    -> Acc: %.4f | Loss: %.4f | Prec: %.4f | Rec: %.4f | F1: %.4f | AUC: %.4f",
        test_rfe_metrics["accuracy"], test_rfe_metrics["loss"], test_rfe_metrics["precision"],
        test_rfe_metrics["recall"], test_rfe_metrics["f1_score"], test_rfe_metrics["roc_auc"]
    )
    logger.info(
        "Train Lasso -> Acc: %.4f | Loss: %.4f | Prec: %.4f | Rec: %.4f | F1: %.4f | AUC: %.4f",
        train_lasso_metrics["accuracy"], train_lasso_metrics["loss"],
        train_lasso_metrics["precision"], train_lasso_metrics["recall"],
        train_lasso_metrics["f1_score"], train_lasso_metrics["roc_auc"]
    )
    logger.info(
        "Test Lasso  -> Acc: %.4f | Loss: %.4f | Prec: %.4f | Rec: %.4f | F1: %.4f | AUC: %.4f",
        test_lasso_metrics["accuracy"], test_lasso_metrics["loss"],
        test_lasso_metrics["precision"], test_lasso_metrics["recall"],
        test_lasso_metrics["f1_score"], test_lasso_metrics["roc_auc"]
    )

    return pd.DataFrame([
        {"method": "rfe",   **test_rfe_metrics},
        {"method": "lasso", **test_lasso_metrics},
    ])


# Main

In [ ]:
def main():
    parser = argparse.ArgumentParser(
        prog="final project",
        description="Data Analysis final project",
    )
    parser.add_argument(
        "--version",
        action="version",
        version=__version__,
    )
    parser.add_argument(
        "--config",
        type=str,
        default="config.json",
        help="Path to the JSON configuration file.",
    )
    parser.add_argument(
        "--evaluate",
        type=str,
        default=None,
        help="Run evaluation on the test set instead of training.",
    )
    args = parser.parse_args()

    config_path = Path(args.config)

    # load configuration and data
    config = load_config_json(config_path)

    file_path = Path(config["data"]["file_path"])
    try:
        df = pd.read_csv(file_path)
    except FileNotFoundError:
        logger.error("Data file not found: %s", file_path)
        raise

    # preprocessing    
    X_cv, X_test, y_cv, y_test = run_preprocess(df, config)
    logger.info(
        "Entries: CV=%s | Test=%s",
        f"{len(X_cv):,}",
        f"{len(X_test):,}",
    )

    # data statistics plots
    plot_flag = config["output"].get("save_plots", False)
    plot_dir = Path(config["output"].get("plots_dir", "outputs/plots"))
    if plot_flag:
        plot_correlations(
            df         = df,
            output_dir = plot_dir,
        )
        plot_norm_stats(
            norm_stats    = compute_normalization_stats(X_cv),
            feature_names = X_cv.columns.tolist(),
            output_dir    = plot_dir,
            max_features  = len(X_cv.columns.tolist()),
        )

    # cross-validation parameters
    n_splits  = config["training"].get("n_splits", 5)
    n_repeats = config["training"].get("n_repeats", 10)
    genes     = (config["training"].get("genes", 30)
                 if isinstance(config["training"].get("genes", 30), list)
                 else [config["training"].get("genes", 30)])
    alpha     = (config["training"].get("alpha", 0.05)
                 if isinstance(config["training"].get("alpha", 0.05), list)
                else [config["training"].get("alpha", 0.05)])
    C         = (config["training"].get("C", 0.1)
                 if isinstance(config["training"].get("C", 0.1), list)
                else [config["training"].get("C", 0.1)])

    timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
    runs_dir  = Path(config["output"].get("runs_dir", "outputs/runs"))
    run_dir   = runs_dir / timestamp
    run_dir.mkdir(parents=True, exist_ok=True)
    # selection bias experiment (Ambroise & McLachlan, 2002)
    if not args.evaluate:
        results, inclusion_probs = run_bias_experiment(
            X_cv, y_cv,
            n_splits   = n_splits,
            n_repeats  = n_repeats,
            parameters = [genes, alpha, C],
        )

        best_run_index = np.argmax(results["unbiased_rfe_val_f1_score"])
        best_inclusion_prob = inclusion_probs[best_run_index]
        logger.info("Best index: %d | Best parameters: [%s %s %s]",
                    best_run_index, results["n_genes"].iloc[best_run_index],
                    results["alpha"].iloc[best_run_index], results["C"].iloc[best_run_index])

        results.to_csv(run_dir / "bias_experiment_results.csv", index=False)

        feature_names = X_cv.columns.tolist()
        try:
            rfe_probs = best_inclusion_prob[0]
        except (IndexError, TypeError):
            rfe_probs = [np.nan] * len(feature_names)
        try:
            lasso_probs = best_inclusion_prob[2]
        except (IndexError, TypeError):
            lasso_probs = [np.nan] * len(feature_names)
        try:
            fdr_probs = best_inclusion_prob[1]
        except (IndexError, TypeError):
            fdr_probs = [np.nan] * len(feature_names)

        df_selected = pd.DataFrame({
            "feature": feature_names,
            "rfe_inclusion_prob":   list(rfe_probs),
            "lasso_inclusion_prob": list(lasso_probs),
            "fdr_inclusion_prob":   list(fdr_probs)
        })
        
        df_selected = df_selected[
            (df_selected["rfe_inclusion_prob"] != 0) | 
            (df_selected["lasso_inclusion_prob"] != 0)
        ]

        df_selected.to_csv(run_dir / "selected_features.csv", index=False)
        best_inclusion_prob = df_selected
    
    else:
        latest_csv = Path(args.evaluate) / "bias_experiment_results.csv"
        results = pd.read_csv(latest_csv)
        best_run_index = np.argmax(results["unbiased_rfe_val_f1_score"])
        best_inclusion_prob = pd.read_csv(latest_csv.parent / "selected_features.csv")

    if plot_dir is not None:
        plot_bias_comparison(
            results=results,
            output_dir=plot_dir,
        )
        plot_inclusion_probabilities(
            rfe_inclusion_prob   = best_inclusion_prob["rfe_inclusion_prob"].values,
            fdr_inclusion_prob   = best_inclusion_prob["fdr_inclusion_prob"].values,
            lasso_inclusion_prob = best_inclusion_prob["lasso_inclusion_prob"].values,
            output_dir = plot_dir,
        )

    # evaluation
    test_results = evaluate(
        X_cv,   y_cv,
        X_test, y_test,
        n_genes              = results["n_genes"].iloc[best_run_index],
        rfe_inclusion_prob   = best_inclusion_prob["rfe_inclusion_prob"].values,
        lasso_inclusion_prob = best_inclusion_prob["lasso_inclusion_prob"].values,
        plot_dir = plot_dir,
        run_dir  = run_dir,
    )
    
    test_results.to_csv(run_dir / "test_results.csv", index=False)


if __name__ == "__main__":
    main()


In [ ]:
! python -m src.final_project.final_project --config configs/config.json